# Regressão Linear com PIB e Índice ABCR

Análise da relação entre o índice de volume do PIB e o Índice ABCR no período de 2006 a 2025.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
pib = pd.read_csv("PIB_IBGE_SIDRA_FINAL.csv", sep=";", decimal=",", header=None, names=["Periodo", "PIB_indice"])
abcr = pd.read_csv("ABRC_FINAL.csv")

In [ ]:
pib["Ano"] = pib["Periodo"].str.extract(r"(\d{4})").astype(int)
pib_anual = pib.groupby("Ano", as_index=False)["PIB_indice"].mean()

abcr["Ano"] = pd.to_datetime(abcr["DATA"], format="%b-%y").dt.year
abcr_anual = abcr.groupby("Ano", as_index=False)["TOTAL"].mean()
abcr_anual = abcr_anual.rename(columns={"TOTAL": "ABCR_indice"})

base = pd.merge(pib_anual, abcr_anual, on="Ano")
base

In [ ]:
plt.scatter(base["PIB_indice"], base["ABCR_indice"])
plt.xlabel("PIB_indice")
plt.ylabel("ABCR_indice")
plt.title("PIB x Índice ABCR")
plt.show()

In [ ]:
correlacao = base["PIB_indice"].corr(base["ABCR_indice"])
print("Correlação:", correlacao)

A correlação mostra a relação entre os dois indicadores. Uma correlação alta não demonstra uma relação de causa e efeito.

In [ ]:
X = base[["PIB_indice"]]
y = base["ABCR_indice"]

X_train = X.iloc[:16]
X_test = X.iloc[16:]
y_train = y.iloc[:16]
y_test = y.iloc[16:]

modelo = LinearRegression()
modelo.fit(X_train, y_train)
y_previsto = modelo.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_previsto)
mse = mean_squared_error(y_test, y_previsto)
r2 = r2_score(y_test, y_previsto)

print("MAE:", mae)
print("MSE:", mse)
print("R²:", r2)

In [ ]:
resultados = pd.DataFrame({
    "Ano": base["Ano"].iloc[16:].values,
    "ABCR_observado": y_test.values,
    "ABCR_previsto": y_previsto
})

resultados

O MAE representa o erro absoluto médio das previsões. O MSE representa a média dos erros ao quadrado. O R² indica o quanto o modelo explica a variação dos dados de teste.

## Dificuldades encontradas

As bases possuem frequências diferentes. O PIB possui dados trimestrais e o Índice ABCR possui dados mensais. Para organizar os dados, foi calculada a média dos quatro trimestres do PIB e dos doze meses do Índice ABCR para cada ano.

## Conclusão

A análise permite verificar a relação entre o índice de volume do PIB e o fluxo de veículos nas rodovias entre 2006 e 2025. A correlação indica o grau de relação entre os indicadores, enquanto as métricas MAE, MSE e R² permitem avaliar as previsões da regressão linear. A correlação encontrada não representa, por si só, uma relação de causa e efeito.